In [1]:
%%capture
%run ../part_1/part1.ipynb
import math
from collections import Counter
from array import array
from collections import defaultdict

In [2]:
#We create a dictionary called field_terms for organizing the searchable terms separately for each field.
field_terms = {
    "title":       list(df["title_terms"]),
    "description": list(df["description_terms"]),
    "category":    list(df["category_terms"]),     
    "brand":       list(df["brand_terms"]),
    "seller":      list(df["seller_terms"]),
    "details":     [build_terms(details_to_text(x)) for x in df["product_details"]],   # values only
}

pids = list(df["pid"])
N = len(df)

Part 1: Indexing 

In [3]:
def create_index(all_terms):
    """
    Implement the inverted index of one field

    Argument:
    all_terms - list of words of all the term of all documents in that specific field

    Returns:
    index - the inverted index (implemented through a Python dictionary) containing terms as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    """
    index = defaultdict(list)

    for doc_id, terms in enumerate(all_terms):
        current_doc_index = {}  # Create temporary index for the current document

        for position, term in enumerate(terms):
            try:
                # if the term is already in the index for the current page (current_page_index)
                # append the position to the corresponding list (elemento 1 del arreglo, el 0 es la id del documento)
                current_doc_index[term][1].append(position)
            except:
                # Add the new term as dict key and initialize the array of positions and add the position
                current_doc_index[term] = [doc_id, array('I', [position])] #'I' indicates unsigned int (int in Python)

        #merge the current page index with the main index
        for term, posting in current_doc_index.items():
            index[term].append(posting)

    return index


In [4]:
# one inverted index per fiel
index = {}
for field in field_terms:
    index[field] = create_index(field_terms[field])
    print(field, ":", len(index[field]), "terms")

title : 738 terms
description : 4698 terms
category : 40 terms
brand : 402 terms
seller : 687 terms
details : 17812 terms


In [5]:
#We have decided the important of these fileds based on the result of part 1 validate labels and the result of the index
weights = {"title": 3.0,"description": 1.5,"category": 0.5,"brand": 1.0, "seller": 0.0, "details": 2.0}

In [6]:
def search(query, top_k=20):
    """
    The output is the list of documents that contain any of the query terms in every field.
    So, we will get the list of documents for each query term, and take the union of them.
    """
    #Normalize the query, so it matches with the pre-processed documents and fields
    query_terms = list(set(build_terms(query)))
    if len(query_terms) == 0:
        return []

    #Find which documents contain each term

    tf = {} #frequency 
    term_docs = {}   # pid of the docuemtns that contain the word                   
    for term in query_terms:
        term_docs[term] = set()
        for field in index:
            tf[(field, term)] = {} #empty dictionary to store the frequency of the current word in the current field
            for posting in index[field].get(term, []):
                doc_id = posting[0]
                positions = posting[1]
                tf[(field, term)][doc_id] = len(positions)
                term_docs[term].add(doc_id)
        if not term_docs[term]:
            return []
        

    # AND, calculates the interseccion
    docs = None
    for term in query_terms:
        if len(term_docs[term]) == 0:
            return []
        if docs is None:
            docs = term_docs[term]
        else:
            docs = docs.intersection(term_docs[term])

    # TF-IDF score
    results = []
    for doc_id in docs:
        score = 0
        for term in query_terms:
            idf = math.log(N / len(term_docs[term]))
            for field in index:
                if doc_id in tf[(field, term)]:
                    score += weights[field] * (1 + math.log(tf[(field, term)][doc_id])) * idf
        results.append((doc_id, score))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]


In [7]:
# Most popular terms
doc_freq = Counter()
for field in index:
    for term, postings in index[field].items():
        doc_freq[term] += len(postings)
print([term for term, count in doc_freq.most_common(40)])

['shirt', 'cloth', 'neck', 'cotton', 'accessori', 'wear', 'men', 'women', 'sleev', 'wash', 'solid', 'print', '1', 'round', 'western', 'regular', 'machin', 'topwear', 'blend', 'fit', 'india', 'casual', 'blue', 'black', 'full', 'pack', 'half', 'slim', 'comfort', 'design', 'fabric', 'polo', 'white', 'short', 'multicolor', 'color', 'gentl', 'pure', '2', 'look']


In [8]:
def show(query, top_k=10):
    """Print the top_k results of a query."""
    for doc_id, score in search(query, top_k):
        print(round(score, 3), pids[doc_id], df["title"].iat[doc_id], "|", df["brand"].iat[doc_id])

show("women full sleeve sweatshirt cotton")

46.436 SWSFFVK89VKGHBZN Full Sleeve Graphic Print Women Sweatshirt | CupidSto
46.436 SWSFFVK8RYHBBFAH Full Sleeve Graphic Print Women Sweatshirt | CupidSto
46.13 SWSFFVKBCQG5FHPF Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBH5YEGFFN Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBB8FHEB7R Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBCY7RSGYJ Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBQUWH72KH Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBYG9ZUBTW Full Sleeve Printed Women Sweatshirt | CupidSto
46.13 SWSFFVKBF5U5AHBR Full Sleeve Printed Women Sweatshirt | CupidSto
45.184 SWSFWEF3XGHFBCJC Full Sleeve Graphic Print Women Sweatshirt | Free Authori


In [9]:
test_queries = {
    1: "men round neck cotton shirt",
    2: "women printed topwear",
    3: "men slim fit jeans",
    4: "women cotton dress",
    5: "men solid polo shirt",
}

for qid, q in test_queries.items():
    print(qid, q, "->", len(search(q, top_k=100000)), "documents")

1 men round neck cotton shirt -> 3227 documents
2 women printed topwear -> 3820 documents
3 men slim fit jeans -> 212 documents
4 women cotton dress -> 517 documents
5 men solid polo shirt -> 919 documents


In [10]:
for qid, q in test_queries.items():
    print(f"\nQuery {qid}: {q}")
    show(q, top_k=5)


Query 1: men round neck cotton shirt
32.138 TSHFUXPWUR2ZTEE2 Color Block Men V Neck Dark Blue T-Shirt | EYETWIST
29.256 TSHFXBTHRSAFFFZP Printed Men Round Neck Grey T-Shirt | dream o
28.573 TSHFWBWEDVTTGYFG Solid Men Round Neck Multicolor T-Shirt  (Pack of 3) | Scott Internation
28.573 TSHFWAWKZ89QFZSQ Solid Men Round Neck Multicolor T-Shirt  (Pack of 5) | Scott Internation
28.573 TSHFWAWKHA7VEBDW Solid Men Round Neck Black T-Shirt  (Pack of 5) | Scott Internation

Query 2: women printed topwear
16.309 TSHFUXQAGVXZHSZW Graphic Print Women Round Neck Black T-Shirt | PixF
16.309 TSHFSR87KZ4FTJHB Printed Women Round Neck Black T-Shirt | PixF
16.309 TSHFUZ6TEJWDPRTA Printed Women Round Neck Yellow T-Shirt | PixF
16.309 TSHFUZ6UTHDHM54F Printed Women Round Neck Red T-Shirt | PixF
16.309 TSHFSR8739SSTDFK Printed Women Round Neck Black T-Shirt | PixF

Query 3: men slim fit jeans
46.86 JEAFEC2HCNHW8V3K Slim Men Multicolor Jeans  (Pack of 2) | NEBRAS
46.86 JEAFDZRVABZZ2PQ3 Stretchable Slim Men

Part 2: Evaluation

In [11]:
import numpy as np
import pandas as pd

1. Evaluation metrics

We code the functions of the evaluation metrics

In [12]:
def precision_at_k(y_true, k=10):
    """P@K = (#relevant docs in the top K)/K"""
    y_true = np.asarray(y_true, dtype=float) #convert to numpy array
    top = y_true[:k] #select the top k elements of the array
    return float(np.sum(top) / k) 

def recall_at_k(y_true, total_relevant, k=10):
    """R@K = (#relevant docs in the top K) / (#relevant docs for the query)"""

    if total_relevant == 0: #to avoid division by 0
        return 0.0

    y_true = np.asarray(y_true, dtype=float) #convert to numpy array
    top = y_true[:k] #select the top k elements of the array

    return float(np.sum(top) / total_relevant)

def f1_at_k(y_true, total_relevant, k=10):
    """F@K = harmonic mean of P@K and R@K"""
    p = precision_at_k(y_true, k) #precision@k
    r = recall_at_k(y_true, total_relevant, k) #recall@k
    if (p+r) == 0: #avoid division by 0
        return 0.0
    
    return float(2*p*r/(p+r))

def avg_precision_at_k(y_true, k=10):
    """
    Parameters
    ----------
    y_true: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    average precision @k : float
    """

    y_true = np.asarray(y_true, dtype=float)

    prec_at_i_list = [] #list to store the computed precision at each relevant document in the top k
    number_of_relevant = 0 #count of relevant documents found
    number_to_iterate = min(k, len(y_true)) #number of documents to iterate over

    for i in range(number_to_iterate): #iterate over the top k documents
        if y_true[i] == 1: #only consider relevant documents
            number_of_relevant += 1
            prec_at_i = number_of_relevant / (i + 1) #accumulated precision at the current position
            prec_at_i_list.append(prec_at_i) #store the precision

    if number_of_relevant == 0:
        return 0.0
    else:
        return float(np.sum(prec_at_i_list) / number_of_relevant) #average precision of all stored precisions

def rr_at_k(y_true, k=10):
    """
    Parameters
    ----------
    y_true: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    Reciprocal Rank for current query
    """

    for i, relevance in enumerate(y_true[:k]): #iterate over the top k documents (labels and positions)
        if relevance == 1: #when it finds the first relevant document, it returns the reciprocal of its rank (position + 1)
            return float(1.0 / (i + 1))

    return 0.0

def dcg_at_k(y_true, k=10):
    y_true = np.asarray(y_true, dtype=float)

    gain = 2 ** y_true[:k] - 1  #compute the gain of relevance (numerator of the formula)
    discounts = np.log2(np.arange(len(gain)) + 2)  #compute the discount factor (denominator of the formula)
    return float(np.sum(gain / discounts))  #return dcg@k

def ndcg_at_k(y_true, k=10, n_relevant=None):
    y_true = np.asarray(y_true, dtype=float)
    if n_relevant is None: #if the number of relevant documents is not provided, we compute it from the ground truth labels
        dcg_max = dcg_at_k(np.sort(y_true)[::-1], k) # ideal: the retrieved docs sorted by their true relevance
    else:
        ideal = np.ones(int(min(n_relevant, k))) # ideal: all the known relevant docs first (binary labels)
        dcg_max = dcg_at_k(ideal, k)
    if dcg_max == 0:
        return 0.0
    return float(dcg_at_k(y_true, k) / dcg_max)

def map_at_k(all_y_true, k=10):
    '''
    Parameters
    ----------
    all_y_true: list of ground truth labels for each query.
    k: number of documents to consider.

    Returns
    -------
    mean average precision @k : float

    Returns
    -------
    mean average precision @k : float
    '''
    avp = []
    for y_true in all_y_true: #iterate over the list of ground truth labels for each query
        avp.append(avg_precision_at_k(y_true, k)) #store the average precision for each query in the list

    if len(avp) == 0:
        return 0.0, []

    return float(np.mean(avp)), avp #return the mean average precision and the list of average precisions

def mrr_at_k(per_query_rr):
    """MRR = mean of the RR@K of every query."""
    return float(np.mean(per_query_rr)) 



2. Evaluation on the predefined queries

In [13]:
labels_df = pd.read_csv("../../data/validation_labels.csv")
validation_queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue"
}

print(labels_df.groupby(["query_id", "labels"]).size())

query_id  labels
1         0          7
          1         13
2         0         10
          1         10
dtype: int64


In [14]:
def get_evaluation_data(query_id, query, k=10):

    query_labels = labels_df[labels_df["query_id"] == query_id] #filter to get the labels for the current query

    #ground truth from validation_labels.csv
    relevance_dict = dict(zip(query_labels["pid"], query_labels["labels"])) 

    #retrieve ranked documents
    results = search(query, top_k=len(pids))

    #keep only documents included in validation_labels.csv
    retrieved_pids = [pids[doc_id] for doc_id, score in results if pids[doc_id] in relevance_dict][:k]

    #relevance labels in ranking order
    y_true = [int(relevance_dict[pid]) for pid in retrieved_pids]

    total_relevant = int((query_labels["labels"] == 1).sum())

    return y_true, total_relevant, retrieved_pids

We prove the function

In [15]:
for query_id, query in validation_queries.items(): #iterate over the validation queries

    y_true, total_relevant, evaluated_pids = get_evaluation_data(query_id, query, k=10) #get the ground truth labels, total relevant documents, and evaluated product ids for the current query

    print("Query:", query)
    print("Relevance labels:", y_true)
    print("Evaluated products retrieved:", len(evaluated_pids))
    print("Total relevant:", total_relevant)
    print()

Query: women full sleeve sweatshirt cotton
Relevance labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Evaluated products retrieved: 10
Total relevant: 13

Query: men slim jeans blue
Relevance labels: [1, 1, 1, 0, 1, 1, 1, 0]
Evaluated products retrieved: 8
Total relevant: 10



In [16]:
all_y_true = []
per_query_rr = []

for query_id, query in validation_queries.items(): #for each query in the validation set, we will compute the evaluation metrics

    y_true, total_relevant, retrieved_pids = get_evaluation_data(query_id, query, k=10) #evaluate the query and get the ground truth labels, total relevant documents, and retrieved product ids

    #compute evaluation metrics
    precision = precision_at_k(y_true, k=10)
    recall = recall_at_k(y_true, total_relevant, k=10)
    avg_precision = avg_precision_at_k(y_true, k=10)
    f1 = f1_at_k(y_true, total_relevant, k=10)
    rr = rr_at_k(y_true, k=10)
    ndcg = ndcg_at_k(y_true, k=10, n_relevant=total_relevant)

    #store results to calculate MAP and MRR
    all_y_true.append(y_true)
    per_query_rr.append(rr)

    print(f"Query {query_id}:")
    print(f"Precision@10 = {precision:.3f}")
    print(f"Recall@10 = {recall:.3f}")
    print(f"Average Precision@10 = {avg_precision:.3f}")
    print(f"F1@10 = {f1:.3f}")
    print(f"RR@10 = {rr:.3f}")
    print(f"NDCG@10 = {ndcg:.3f}")

#compute MAP and MRR for all queries
map_score, ap_values = map_at_k(all_y_true, k=10)
mrr_score = mrr_at_k(per_query_rr)

print(f"MAP@10 = {map_score:.3f}")
print(f"MRR@10 = {mrr_score:.3f}")

Query 1:
Precision@10 = 1.000
Recall@10 = 0.769
Average Precision@10 = 1.000
F1@10 = 0.870
RR@10 = 1.000
NDCG@10 = 1.000
Query 2:
Precision@10 = 0.600
Recall@10 = 0.600
Average Precision@10 = 0.915
F1@10 = 0.600
RR@10 = 1.000
NDCG@10 = 0.706
MAP@10 = 0.958
MRR@10 = 1.000


3. Establishing the ground truth for each document and test queries

In [17]:
for query_id, query in test_queries.items(): #iterate over the test queries

    results = search(query, top_k=10) #retrieve the top 10 results for the current query

    print(f"Query {query_id}: {query}")

    for rank, (doc_id, score) in enumerate(results, start=1):
        print(f"Rank: {rank}")
        print(f"PID: {pids[doc_id]}")
        print(f"TF-IDF score: {score:.4f}")

Query 1: men round neck cotton shirt
Rank: 1
PID: TSHFUXPWUR2ZTEE2
TF-IDF score: 32.1376
Rank: 2
PID: TSHFXBTHRSAFFFZP
TF-IDF score: 29.2563
Rank: 3
PID: TSHFWBWEDVTTGYFG
TF-IDF score: 28.5730
Rank: 4
PID: TSHFWAWKZ89QFZSQ
TF-IDF score: 28.5730
Rank: 5
PID: TSHFWAWKHA7VEBDW
TF-IDF score: 28.5730
Rank: 6
PID: TSHFWAWKRWEDMYJ8
TF-IDF score: 28.5730
Rank: 7
PID: TSHFWBWESMCTBUKC
TF-IDF score: 28.5730
Rank: 8
PID: TSHFWBWEQMAJDPAT
TF-IDF score: 28.5730
Rank: 9
PID: TSHFWAWKG67G9FUX
TF-IDF score: 28.5730
Rank: 10
PID: TSHFWBWEZHVQEHYZ
TF-IDF score: 28.5730
Query 2: women printed topwear
Rank: 1
PID: TSHFUXQAGVXZHSZW
TF-IDF score: 16.3095
Rank: 2
PID: TSHFSR87KZ4FTJHB
TF-IDF score: 16.3095
Rank: 3
PID: TSHFUZ6TEJWDPRTA
TF-IDF score: 16.3095
Rank: 4
PID: TSHFUZ6UTHDHM54F
TF-IDF score: 16.3095
Rank: 5
PID: TSHFSR8739SSTDFK
TF-IDF score: 16.3095
Rank: 6
PID: TSHFSPXRZUGUZ3BR
TF-IDF score: 16.3095
Rank: 7
PID: TSHFSPXRPMYDX4MH
TF-IDF score: 16.3095
Rank: 8
PID: TSHFSPXRZHVZRRDH
TF-IDF score: 16.

Create a dataframe for the evaluated documents. It helps to extract the information of the documents retrieved by our search system

In [18]:
evaluation_rows = []
for query_id, query in test_queries.items():
    results = search(query, top_k=10) #retrieve the top 10 results for the current query

    for rank, (doc_id, score) in enumerate(results, start=1):
        pid = pids[doc_id]

        evaluation_rows.append({
            "query_id": query_id,
            "query": query,
            "rank": rank,
            "pid": pid,
            "tfidf_score": score
        })

evaluation_df = pd.DataFrame(evaluation_rows)

#add product information to the evaluation dataframe
evaluation_df = evaluation_df.merge(
    df[["pid", "title", "description", "brand", "category", "sub_category"]], on="pid", how="left", validate="many_to_one")

#column for manual relevance judgments
evaluation_df["relevance"] = pd.NA

display(evaluation_df[
    ["query_id", "rank", "pid", "title", "description", "relevance"]
])

,query_id,rank,pid,title,description,relevance
0,1,1,TSHFUXPWUR2ZTEE2,Color Block Men V Neck Dark Blue T-Shirt,1.PC cotton bio-washed full sleeves t-shirts. ...,<NA>
1,1,2,TSHFXBTHRSAFFFZP,Printed Men Round Neck Grey T-Shirt,Express your individuality and thoughts with D...,<NA>
2,1,3,TSHFWBWEDVTTGYFG,Solid Men Round Neck Multicolor T-Shirt (Pack...,Refresh your wardrobe with the Premium Collect...,<NA>
3,1,4,TSHFWAWKZ89QFZSQ,Solid Men Round Neck Multicolor T-Shirt (Pack...,Refresh your wardrobe with the Premium Collect...,<NA>
4,1,5,TSHFWAWKHA7VEBDW,Solid Men Round Neck Black T-Shirt (Pack of 5),Refresh your wardrobe with the Premium Collect...,<NA>
5,1,6,TSHFWAWKRWEDMYJ8,Solid Men Round Neck White T-Shirt (Pack of 5),Refresh your wardrobe with the Premium Collect...,<NA>
6,1,7,TSHFWBWESMCTBUKC,Solid Men Round Neck Dark Blue T-Shirt (Pack ...,Refresh your wardrobe with the Premium Collect...,<NA>
7,1,8,TSHFWBWEQMAJDPAT,Solid Men Round Neck White T-Shirt (Pack of 3),Refresh your wardrobe with the Premium Collect...,<NA>
8,1,9,TSHFWAWKG67G9FUX,Solid Men Round Neck Grey T-Shirt (Pack of 5),Refresh your wardrobe with the Premium Collect...,<NA>
9,1,10,TSHFWBWEZHVQEHYZ,Solid Men Round Neck Grey T-Shirt (Pack of 3),Refresh your wardrobe with the Premium Collect...,<NA>


For every query, print the top 10 docs (title, descriptions...), and with the results, we judge if they are relevant or not

In [19]:
for _, row in evaluation_df[evaluation_df["query_id"] == 1].iterrows():
    print(f"Rank {row['rank']}: {row['title']}")
    print(f"Description: {row['description']}")

Rank 1: Color Block Men V Neck Dark Blue T-Shirt
Description: 1.PC cotton bio-washed full sleeves t-shirts. 2. High quality 30 s bio-washed cotton fabric 3. 180 gsm 4. Soft and wrinkle free fabric. COTTON BIO WASHED MAROON BLACK WAIST COAT T-SHIRTS . HIGH QUALITY SMOOTH FABRIC .AND QUALITY PRODUCT AND UNIQUE STYLE T-SHIRT. AVAILABLE IN ALL SIZES FROM, S-XXL. t-shirts combo for men t-shirts for women stylish latest t-shirts for men Search Terms ??? Blended Designer V-neck O-neck Round Neck zipper hooded Hoodie Polo Collar Henley Button Bio-washed T-shirt Dress clothes clothing apparel t-shirts combo for men t-shirts for women stylish latest t-shirts for men maroon black t shirt 8.8 men hooded maroon black t shirt 8.3 men maroon & black striped round neck t-shirt 8.3 solid men hooded black, maroon t-shirt 8.0 fresh trend maroon black white cotton round neck t shirt for women 8.0 fresh trend maroon black white t-shirt for woman 8.0 fresh trend maroon white black t shirts for women 8.0 mar

The first result has neck V so it is not relevant. The other ones, have in their titles round neck and cotton.

In [20]:
labels_query_1 = [0, 1, 1, 1, 1, 1, 1, 1, 1, 1] #manual relevance judgments for query 1 (0 = not relevant, 1 = relevant)

indices = evaluation_df.loc[evaluation_df["query_id"] == 1].sort_values("rank").index #get the indices of the rows corresponding to query 1, sorted by rank

evaluation_df.loc[indices, "relevance"] = labels_query_1 #append the manual relevance judgments to the "relevance" column of the evaluation dataframe for query 1

In [21]:
for _, row in evaluation_df[evaluation_df["query_id"] == 2].iterrows():
    print(f"Rank {row['rank']}: {row['title']}")
    print(f"Description: {row['description']}")

Rank 1: Graphic Print Women Round Neck Black T-Shirt
Description: Refresh your clothing with this latest new Printed stylish T Shirt for Women Women. Combo it with Jeans, Trouser, Track Pant, Bottoms for the perfect casual daily wear. These graphic tshirts are Perfect for Gym, Workout, Running, Sport, Partywear, and Casual Wear for every range of age groups from Girls, Girls, and Ladies.
Rank 2: Printed Women Round Neck Black T-Shirt
Description: Refresh your clothing with this latest new Printed stylish T Shirt for Women Women. Combo it with Jeans, Trouser, Track Pant, Bottoms for the perfect casual daily wear. These graphic tshirts are Perfect for Gym, Workout, Running, Sport, Partywear, and Casual Wear for every range of age groups from Girls, Girls, and Ladies.
Rank 3: Printed Women Round Neck Yellow T-Shirt
Description: Refresh your clothing with this latest new Printed stylish T Shirt for Women Women. Combo it with Jeans, Trouser, Track Pant, Bottoms for the perfect casual daily 

All products are top wear for women and have printed or graphic print.

In [22]:
labels_query_2 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

indices = evaluation_df.loc[evaluation_df["query_id"] == 2].sort_values("rank").index

evaluation_df.loc[indices, "relevance"] = labels_query_2

In [23]:
for _, row in evaluation_df[evaluation_df["query_id"] == 3].iterrows():
    print(f"Rank {row['rank']}: {row['title']}")
    print(f"Description: {row['description']}")

Rank 1: Slim Men Multicolor Jeans  (Pack of 2)
Description: Authentic Mens Slim Fit Jean. This jean is constructed with durable materials built for long-lasting comfort. These jeans sit below the waist with a slim fit from hip to ankle. This pair has just the right amount of stretch for all-day comfort. Cut close to the body, the Nebraska Slim Jeans is not just a jean but a style statement with no compromise on comfort. .Arch detailing at back pockets/patch pockets, contrast stitch details, waist band with belt loops and brand patch give it a more attractive look. The narrow leg also means endless style options. For a laid-back daytime look, try a slightly scrunched leg and sneakers. Once night rolls around, try a 2-inch cuff with a Chelsea boot.
Rank 2: Stretchable Slim Men Blue Jeans  (Pack of 2)
Description: Authentic Mens Slim Fit Jean. This jean is constructed with durable materials built for long-lasting comfort. These jeans sit below the waist with a slim fit from hip to ankle. 

All products are slim fit jeans for men as we can see in their titles and descriptions.

In [24]:
labels_query_3 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

indices = evaluation_df.loc[evaluation_df["query_id"] == 3].sort_values("rank").index

evaluation_df.loc[indices, "relevance"] = labels_query_3

In [25]:
for _, row in evaluation_df[evaluation_df["query_id"] == 4].iterrows():
    print(f"Rank {row['rank']}: {row['title']}")
    print(f"Description: {row['description']}")

Rank 1: Solid Women V Neck Maroon T-Shirt
Description: 1.PC cotton bio-washed full sleeves t-shirts. 2. High quality 30 s bio-washed cotton fabric 3. 180 gsm 4. Soft and wrinkle free fabric. COTTON BIO WASHED MAROON BLACK WAIST COAT T-SHIRTS . HIGH QUALITY SMOOTH FABRIC .AND QUALITY PRODUCT AND UNIQUE STYLE T-SHIRT. AVAILABLE IN ALL SIZES FROM, S-XXL. t-shirts combo for women t-shirts for women stylish latest t-shirts for women Search Terms ??? Blended Designer V-neck O-neck Round Neck zipper hooded Hoodie Polo Collar Henley Button Bio-washed T-shirt Dress clothes clothing apparel t-shirts combo for women t-shirts for women stylish latest t-shirts for women maroon black t shirt 8.8 women hooded maroon black t shirt 8.3 women maroon & black striped round neck t-shirt 8.3 solid women hooded black, maroon t-shirt 8.0 fresh trend maroon black white cotton round neck t shirt for women 8.0 fresh trend maroon black white t-shirt for woman 8.0 fresh trend maroon white black t shirts for women 

Since the first result is a T-Shirt and not a dress, we can say that is no relevant. However, the other ones are relevant since are cotton dresses.

In [26]:
labels_query_4 = [0, 1, 1, 1, 1, 1, 1, 1, 1, 1]

indices = evaluation_df.loc[evaluation_df["query_id"] == 4].sort_values("rank").index

evaluation_df.loc[indices, "relevance"] = labels_query_4

In [27]:
for _, row in evaluation_df[evaluation_df["query_id"] == 5].iterrows():
    print(f"Rank {row['rank']}: {row['title']}")
    print(f"Description: {row['description']}")

Rank 1: Solid Men Polo Neck White T-Shirt
Description: Refresh your wardrobe with the new range of Summer Collections of Organic Cotton Polo T-shirts from Scott International. This Polo T-shirts are made of is 100% Pure Premium Organic Cotton fabric, Super Combed, Solid Polo and gives utmost comfort during all temperatures. Durable stitch and finish make this T-shirt a perfect Formal and Casual wear. Contrast Tipping on the Collar and Sleeves makes this Product have a unique look as compared to other products in the Polo segment. Pair it with Jeans or Casual for a perfect Casual or Party look
Rank 2: Solid Men Polo Neck Purple T-Shirt
Description: Refresh your wardrobe with the new range of Summer Collections of Organic Cotton Polo T-shirts from Scott International. This Polo T-shirts are made of is 100% Pure Premium Organic Cotton fabric, Super Combed, Solid Polo and gives utmost comfort during all temperatures. Durable stitch and finish make this T-shirt a perfect Formal and Casual w

The 10 products are relevant since they are polo shirts for men in a solid form.

In [28]:
labels_query_5 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

indices = evaluation_df.loc[evaluation_df["query_id"] == 5].sort_values("rank").index

evaluation_df.loc[indices, "relevance"] = labels_query_5

Calculate and analyze the results with the metrics

In [29]:
test_y_true = {}

for query_id, query in test_queries.items():

    query_results = evaluation_df[evaluation_df["query_id"] == query_id].sort_values("rank")

    y_true = query_results["relevance"].astype(int).tolist()

    test_y_true[query_id] = y_true

    print(f"Query {query_id}: {query}")
    print("Relevance labels:", y_true)

Query 1: men round neck cotton shirt
Relevance labels: [0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Query 2: women printed topwear
Relevance labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Query 3: men slim fit jeans
Relevance labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Query 4: women cotton dress
Relevance labels: [0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Query 5: men solid polo shirt
Relevance labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [30]:
evaluation_results = []
all_y_true = []
per_query_rr = []

for query_id, query in test_queries.items():
    y_true = test_y_true[query_id]

    precision = precision_at_k(y_true, k=10)
    avg_precision = avg_precision_at_k(y_true, k=10)
    rr = rr_at_k(y_true, k=10)
    ndcg = ndcg_at_k(y_true, k=10)

    all_y_true.append(y_true)
    per_query_rr.append(rr)

    print(f"Query {query_id}: {query}")
    print(f"  Precision@10: {precision:.3f}")
    print(f"  AP@10:        {avg_precision:.3f}")
    print(f"  RR@10:        {rr:.3f}")
    print(f"  NDCG@10:      {ndcg:.3f}")
    
map_score, _ = map_at_k(all_y_true, k=10)
mrr_score = mrr_at_k(per_query_rr)

print("Overall evaluation:")
print(f"  MAP@10: {map_score:.3f}")
print(f"  MRR@10: {mrr_score:.3f}")

Query 1: men round neck cotton shirt
  Precision@10: 0.900
  AP@10:        0.786
  RR@10:        0.500
  NDCG@10:      0.833
Query 2: women printed topwear
  Precision@10: 1.000
  AP@10:        1.000
  RR@10:        1.000
  NDCG@10:      1.000
Query 3: men slim fit jeans
  Precision@10: 1.000
  AP@10:        1.000
  RR@10:        1.000
  NDCG@10:      1.000
Query 4: women cotton dress
  Precision@10: 0.900
  AP@10:        0.786
  RR@10:        0.500
  NDCG@10:      0.833
Query 5: men solid polo shirt
  Precision@10: 1.000
  AP@10:        1.000
  RR@10:        1.000
  NDCG@10:      1.000
Overall evaluation:
  MAP@10: 0.914
  MRR@10: 0.800


The evaluation results show that the seach engine performs well on the 5 test queries. Queries 2,3 and 5 have Precicion@10, AP@10, RR@10 and NDCG@10 of 1.000, since the 10 products retrieved are relevant.

Queries 1 and 4 obtain lower results. In both cases, 9 of 10 products are relevant, giving Precision@10 of 0.900. However, since the first product of both queries is not relevant, RR@10 reduces to 0.500, AP@10 reduces to 0.786 and NDCG@10 reduces to 0.833.

This shows that position in the ranking is also important. Although the TF-IDF ranking retrieves mostly relevant products, it sometimes places an irrelevant product above more suitable results.

Overall, the system achieves a MAP@10 og 0.914 and a MRR@10 of 0.800.